In [3]:
import torch.utils.data as data
from pathlib import Path
import torchvision.transforms as transforms
import json
import pandas as pd
import os
# Assuming seek_code.py is in a different directory
import sys
sys.path.append('/data/vision/beery/scratch/antoine/CBM_reid/methods')
from seek_code import SEEK

In [4]:
dataset_dir = Path('/archive/vision/beery/animal_reid/datasets/elephants_zooniverse')
ele_SEEK_code_dir = dataset_dir / 'ele_id_project/data/out_apr2/SEEK_codes_.csv'

# Define the root of Path
root = Path('/data/vision/beery/scratch/antoine/CBM_reid')
# path to the script that generates the image-wise SEEK codes
script_path = root / Path('data_processing/generate_image-wise_SEEK.py')

# Path to the generated image-wise SEEK codes (also in the script itself)
subject_SEEK_code_dir = root / Path('data_processing/data/out_apr2/SEEK_codes_.csv') 

# Path to which you export the data dictionary
output_path = root / Path('data_processing/data/out_apr2/image_dictonary.csv')

processed_dir = root / Path('preprocessed_images')

### The dataset directory is organised as such

The dataset is inherited from Zooniverse. Our code uses in particular `[ear,general,special]_seek_annotations.csv`, `SEEK_codes.csv`, `SEEK_dict.json`, `sub_ele_pairs.json`, `EFA_IDI_complete_manifest_uploaded.json`, `EFA_IDU_complete_manifest_uploaded.json` and the images.


    dataset_dir
        ↳ ele_id_project
            ↳ data
                ↳ out_apr_2
                    ↳ [ear,general,special]_workflow_annotations.csv
                    ↳ [ear,general,special]_seek_annotations.csv
                    ↳ SEEK_codes.csv
                    ↳ SEEK_dict.json
                    ↳ sub_ele_pairs.json
            ↳ data_processing
                ↳ get_seek_annotations.py
                ↳ get_seek_dict.py
                ↳ get_sub_id_ele_id_dict.py
                ↳ get_workflow_annotations.py
                ↳ utils.py
                ↳ workflow_constants_traditional.json
            ↳ Project_documentation.md
            ↳ README.md
        ↳ images
            ↳ EFA_IDI
                ↳ B1000
                    ↳ B1000_R1
                        ↳ EFA_IDI_B1000_R1_IMAG0001.JPG
                        ...
                    ...
                ...
            ↳ EFA_IDU
                ...
        ↳ EFA_IDI_complete_manifest_uploaded.json
        ↳ EFA_IDU_complete_manifest_uploaded.json        

In [5]:
seek_T555 = SEEK('B20T__E8000-0000X0_S00') 

# Testing my SEEK class
print(seek_T555)      
print(seek_T555[1])   
print(seek_T555.age)

B20T__E8000-0000X0_S00
tensor([0., 1.])
20


In [6]:
# Load JSON files of the manifests (dictonary of subjects)
with open(dataset_dir /'EFA_IDU__complete__manifest_uploaded.json', 'r') as file:
    data_idu = json.load(file)

with open(dataset_dir / 'EFA_IDI__complete__manifest_uploaded.json', 'r') as file:
    data_idi = json.load(file)

# Helper function to transform data into a DataFrame, ensuring subject_id is a string
def json_to_df(data):
    rows = []
    for entry in data.values():
        # Check if 'subject_id' exists in the 'info' dictionary
        if 'subject_id' in entry['info']:
            subject_id = str(entry['info']['subject_id'])  # Convert to string
            image = entry['images'][0] if entry['images'] else None
            site = entry['upload_metadata'].get('#site', None)
            season = entry['upload_metadata'].get('#season', None)
            capture = entry['upload_metadata'].get('#capture', None)
            anonymized_capture_id = entry['info'].get('anonymized_capture_id', None)
            
            rows.append([subject_id, image, site, season, capture, anonymized_capture_id])
        else:
            print(f"Missing 'subject_id' in entry: {entry}")  # Log missing subject_id entries if needed
    
    df = pd.DataFrame(rows, columns=['subject_id', 'image', '#site', '#season', '#capture', 'anonymized_capture_id'])
    
    # Handle duplicates by resetting the index
    df['subject_id'] = df['subject_id'].astype(int)
    df.set_index('subject_id', inplace=True)
    df = df[~df.index.duplicated(keep='first')]
    
    return df

# Convert JSON data to DataFrames
df_idu = json_to_df(data_idu)
df_idi = json_to_df(data_idi)

# Concatenate the DataFrames
joined_manifest = pd.concat([df_idu, df_idi])
joined_manifest.head()

Missing 'subject_id' in entry: {'upload_metadata': {'#capture_id': 'EFA_IDI#B1002#1#11', '#site': 'B1002', '#roll': '1', '#season': 'EFA_IDI', '#capture': '11', 'attribution': 'University of Minnesota + Snapshot Safari + Elephants for Africa', 'license': 'University of Minnesota + Snapshot Safari + Elephants for Africa'}, 'info': {'uploaded': False}, 'images': ['EFA_IDI/B1002/B1002_R1/EFA_IDI_B1002_R1_IMAG0011.JPG']}


,image,#site,#season,#capture,anonymized_capture_id
subject_id,,,,,
92126812,EFA_IDU/T1000/T1000_R1/EFA_IDU_T1000_R1_IMAG00...,T1000,EFA_IDU,1,a7f15c6f92cd6028dc590ecf8dfc386476faa3b4
92126813,EFA_IDU/T1000/T1000_R1/EFA_IDU_T1000_R1_IMAG00...,T1000,EFA_IDU,2,4e2838a5c2c0294128f8b72b996c2edfcf498b04
92126814,EFA_IDU/T1001/T1001_R1/EFA_IDU_T1001_R1_IMAG00...,T1001,EFA_IDU,1,da8a85f1548058c930d024bec589fb2cdb484d36
92126815,EFA_IDU/T1001/T1001_R1/EFA_IDU_T1001_R1_IMAG00...,T1001,EFA_IDU,2,9229c652be571b6970dd2ba7fbe104597dcc851e
92126816,EFA_IDU/T1001/T1001_R1/EFA_IDU_T1001_R1_IMAG00...,T1001,EFA_IDU,3,d29ac1e2a0c9da2f40c3f2e1dae60aff100aa042


In [7]:
# Generate the image-specific SEEK codes using the generate_image-wise_SEEK_codes.py script
exec(open(script_path).read())

Upload of all data files complete.
data/out_apr2/SEEK_dict.json
Result dictionary saved as JSON file: data/out_apr2/SEEK_dict.json
sub_ele_pairs: {'92100839': 'B1000', '92100840': 'B1000', '92100841': 'B1000', '92100842': 'B1000', '92100843': 'B1000', '92100844': 'B1000', '92100845': 'B1000', '92100846': 'B1000', '92100847': 'B1000', '92100848': 'B1000', '92100849': 'B1000', '92100850': 'B1000', '92100851': 'B1000', '92100852': 'B1000', '92100853': 'B1000', '92100854': 'B1000', '92100855': 'B1000', '92100856': 'B1000', '92100857': 'B1000', '92100858': 'B1000', '92100859': 'B1000', '92100860': 'B1000', '92100861': 'B1000', '92100862': 'B1000', '92100863': 'B1000', '92100864': 'B1000', '92100865': 'B1000', '92100866': 'B1000', '92100867': 'B1000', '92100868': 'B1000', '92100869': 'B1000', '92100870': 'B1000', '92100871': 'B1000', '92100872': 'B1000', '92100873': 'B1000', '92100874': 'B1000', '92100875': 'B1000', '92100876': 'B1000', '92100877': 'B1000', '92100878': 'B1001', '92100879': '

In [8]:
# Go read the generated image-wise SEEK codes in the output directory specified both at the top of the script and in the script itself
subject_SEEK_code = pd.read_csv(subject_SEEK_code_dir)
subject_SEEK_code.set_index('subject_id', inplace=True)
subject_SEEK_code.dropna(inplace=True)

matching_subject_ids = joined_manifest.index.intersection(subject_SEEK_code.index)
print(f"Number of matching subject_ids: {len(matching_subject_ids)}")

# Make a copy of the joined_manifest
manifest_with_seek = joined_manifest.copy()

# Add a column for the SEEK code, only if it passes the length check
manifest_with_seek['subject-SEEK'] = manifest_with_seek.index.map(
    lambda idx: SEEK(subject_SEEK_code.loc[idx, 'SEEK']) 
    if idx in subject_SEEK_code.index and len(subject_SEEK_code.loc[idx, 'SEEK']) == 22
    else None
)

# Drop rows where SEEK is None
manifest_with_seek.dropna(subset=['subject-SEEK'], inplace=True)
# Rename #site as 'ele_id'
manifest_with_seek.rename(columns={'#site': 'ele_id'}, inplace=True)
# Reset the index to make 'subject_id' a column instead of the index
manifest_with_seek.reset_index(inplace=True)

print(f"Length of manifest with valid subject-SEEK: {len(manifest_with_seek)}")
manifest_with_seek.head()

Number of matching subject_ids: 8573
Length of manifest with valid subject-SEEK: 8501


,subject_id,image,ele_id,#season,#capture,anonymized_capture_id,subject-SEEK
0,92126812,EFA_IDU/T1000/T1000_R1/EFA_IDU_T1000_R1_IMAG00...,T1000,EFA_IDU,1,a7f15c6f92cd6028dc590ecf8dfc386476faa3b4,B00T_1E____-0000X_0S00
1,92126813,EFA_IDU/T1000/T1000_R1/EFA_IDU_T1000_R1_IMAG00...,T1000,EFA_IDU,2,4e2838a5c2c0294128f8b72b996c2edfcf498b04,B00T11E8080-0000X00S00
2,92126815,EFA_IDU/T1001/T1001_R1/EFA_IDU_T1001_R1_IMAG00...,T1001,EFA_IDU,2,9229c652be571b6970dd2ba7fbe104597dcc851e,B00T11E0000-0000X_0S00
3,92126816,EFA_IDU/T1001/T1001_R1/EFA_IDU_T1001_R1_IMAG00...,T1001,EFA_IDU,3,d29ac1e2a0c9da2f40c3f2e1dae60aff100aa042,B00T11E____-0040X_0S00
4,92126817,EFA_IDU/T1001/T1001_R1/EFA_IDU_T1001_R1_IMAG00...,T1001,EFA_IDU,4,f73d21ecc60aaf4e940a02283aaeb3ffb4604cee,C00T1_E____-4000X_0S0_


In [9]:
ele_SEEK_code = pd.read_csv(ele_SEEK_code_dir)

# Convert SEEK codes to SEEK objects
ele_SEEK_code['SEEK'] = ele_SEEK_code['SEEK'].apply(lambda code: SEEK(code) if len(code) == 22 else None)

# Drop rows where SEEK is None
ele_SEEK_code.dropna(subset=['SEEK'], inplace=True)

# Merge the column SEEK into manifest_with_seek according to ele_id but only add the SEEK code (column to the most right)
manifest_with_seek = manifest_with_seek.merge(ele_SEEK_code[['ele_id', 'SEEK']], on='ele_id', how='left', suffixes=('', '_ele'))
manifest_with_seek.rename(columns={'SEEK': 'ele-SEEK'}, inplace=True)

# re-index to subejct_id
#manifest_with_seek.set_index('subject_id', inplace=True)

manifest_with_seek.dropna(subset=['ele-SEEK'], inplace=True)
print(f"Length of manifest with valid ele-SEEK: {len(manifest_with_seek)}")
manifest_with_seek.head()

# Save the updated manifest_with_seek DataFrame to a CSV file

manifest_with_seek.to_csv(output_path)
manifest_with_seek.head()

Length of manifest with valid ele-SEEK: 8501


,subject_id,image,ele_id,#season,#capture,anonymized_capture_id,subject-SEEK,ele-SEEK
0,92126812,EFA_IDU/T1000/T1000_R1/EFA_IDU_T1000_R1_IMAG00...,T1000,EFA_IDU,1,a7f15c6f92cd6028dc590ecf8dfc386476faa3b4,B00T_1E____-0000X_0S00,B00T11E8080-0000X_0S00
1,92126813,EFA_IDU/T1000/T1000_R1/EFA_IDU_T1000_R1_IMAG00...,T1000,EFA_IDU,2,4e2838a5c2c0294128f8b72b996c2edfcf498b04,B00T11E8080-0000X00S00,B00T11E8080-0000X_0S00
2,92126815,EFA_IDU/T1001/T1001_R1/EFA_IDU_T1001_R1_IMAG00...,T1001,EFA_IDU,2,9229c652be571b6970dd2ba7fbe104597dcc851e,B00T11E0000-0000X_0S00,B00T11E____-5000X_0S00
3,92126816,EFA_IDU/T1001/T1001_R1/EFA_IDU_T1001_R1_IMAG00...,T1001,EFA_IDU,3,d29ac1e2a0c9da2f40c3f2e1dae60aff100aa042,B00T11E____-0040X_0S00,B00T11E____-5000X_0S00
4,92126817,EFA_IDU/T1001/T1001_R1/EFA_IDU_T1001_R1_IMAG00...,T1001,EFA_IDU,4,f73d21ecc60aaf4e940a02283aaeb3ffb4604cee,C00T1_E____-4000X_0S0_,B00T11E____-5000X_0S00


## Extracting picture time

We want to make sure that we don't test our model on images that are too similar to images present in the datasets. Images taken in the same encounter might be too similar. Thus, we want to group them by encounter and put them all together in the same split (train, test or validation).

In order to group pictures by "encounters", we extract the picture date and time from the EXIF of the images

In [10]:
from PIL import Image
from PIL.ExifTags import TAGS
from datetime import datetime

manifest_with_seek


def get_exif(image_path):
    image = Image.open(image_path)
    exif_data = image._getexif()
    if exif_data is None:
        return 'No exif data found'
    exif = {TAGS.get(tag, tag): value for tag, value in exif_data.items()}
    date_time = exif.get('DateTime', 'Date not found')
    print('image_path: ', image_path, 'date_time: ', date_time)

    return date_time

manifest_with_seek['picture_time'] = manifest_with_seek['image'].apply(lambda x: get_exif('/archive/vision/beery/animal_reid/datasets/elephants_zooniverse/images/' + x))


image_path:  /archive/vision/beery/animal_reid/datasets/elephants_zooniverse/images/EFA_IDU/T1000/T1000_R1/EFA_IDU_T1000_R1_IMAG0001.JPG date_time:  2015:05:22 21:31:22
image_path:  /archive/vision/beery/animal_reid/datasets/elephants_zooniverse/images/EFA_IDU/T1000/T1000_R1/EFA_IDU_T1000_R1_IMAG0002.JPG date_time:  2015:05:22 21:33:46
image_path:  /archive/vision/beery/animal_reid/datasets/elephants_zooniverse/images/EFA_IDU/T1001/T1001_R1/EFA_IDU_T1001_R1_IMAG0002.JPG date_time:  2015:05:22 21:32:50
image_path:  /archive/vision/beery/animal_reid/datasets/elephants_zooniverse/images/EFA_IDU/T1001/T1001_R1/EFA_IDU_T1001_R1_IMAG0003.JPG date_time:  2015:05:22 21:32:51
image_path:  /archive/vision/beery/animal_reid/datasets/elephants_zooniverse/images/EFA_IDU/T1001/T1001_R1/EFA_IDU_T1001_R1_IMAG0004.JPG date_time:  2015:05:22 21:32:52
image_path:  /archive/vision/beery/animal_reid/datasets/elephants_zooniverse/images/EFA_IDU/T1002/T1002_R1/EFA_IDU_T1002_R1_IMAG0001.JPG date_time:  2015:0

In [11]:
def standardize_datetime(dt_str):
    try:
        return datetime.strptime(dt_str, '%Y:%m:%d %H:%M:%S')
    except ValueError:
        return None

# Standardization of time format
initial_count = len(manifest_with_seek)
manifest_with_seek['picture_time'] = manifest_with_seek['picture_time'].apply(standardize_datetime)

# Drop rows with non-standard date times
manifest_with_seek.dropna(subset=['picture_time'], inplace=True)

print(f"The extraciton of picture time worked on : {len(manifest_with_seek)} rows, and {initial_count - len(manifest_with_seek) } rows were deleted.")
manifest_with_seek.head()
manifest_with_seek.to_csv(output_path)


The extraciton of picture time worked on : 8444 rows, and 57 rows were deleted.


## Pre-processing the images to access them faster

A large amount of the computing time is dedicated to applying the tranform to the images by preprocessing the images, you can access them much faster.


In [12]:
from pathlib import Path
from PIL import Image
import torchvision.transforms as transforms
import pandas as pd  # Ensure pandas is imported if not already
from tqdm import tqdm  # Import tqdm for progress tracking

# Define the export directory for pre-processed images
processed_dir = Path('/data/vision/beery/scratch/antoine/CBM_reid/preprocessed_images')  # Ensure processed_dir is a Path object
processed_dir.mkdir(parents=True, exist_ok=True)

# Define the transformation
transform = transforms.Compose([
    transforms.Resize([224, 224]),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

# Function to preprocess and save the image
def preprocess_and_save(image_path, processed_dir, transform):
    
    #image_path = image_path.with_suffix('.jpg')

    image = Image.open(image_path).convert('RGB')
    image = transform(image)
    

    # Construct the export path and ensure a .jpg extension
    relative_path = Path(*image_path.parts[-4:])  # Take the last four parts of the path
    export_path = processed_dir / Path( image_path).parts[-4] / Path( image_path).parts[-3] / Path( image_path).parts[-1] # Save with .jpg extension

    export_path.parent.mkdir(parents=True, exist_ok=True)  # Create directories if they do not exist
    transforms.ToPILImage()(image).save(export_path, format="JPEG")  # Ensure image is saved as JPEG
    return export_path

# Add a column with the path to the pre-processed image, tracking progress with tqdm
tqdm.pandas(desc="Processing Images")
manifest_with_seek['preprocessed_image_path'] = manifest_with_seek['image'].progress_apply(
    lambda img_path: preprocess_and_save(dataset_dir / 'images' / img_path, processed_dir, transform)
)

# Save the DataFrame to CSV
manifest_with_seek.to_csv(output_path, index=False)
manifest_with_seek.head()


Processing Images:   0%|          | 0/8444 [00:00<?, ?it/s]

Processing Images: 100%|██████████| 8444/8444 [20:01<00:00,  7.03it/s]


,subject_id,image,ele_id,#season,#capture,anonymized_capture_id,subject-SEEK,ele-SEEK,picture_time,preprocessed_image_path
0,92126812,EFA_IDU/T1000/T1000_R1/EFA_IDU_T1000_R1_IMAG00...,T1000,EFA_IDU,1,a7f15c6f92cd6028dc590ecf8dfc386476faa3b4,B00T_1E____-0000X_0S00,B00T11E8080-0000X_0S00,2015-05-22 21:31:22,/data/vision/beery/scratch/antoine/CBM_reid/pr...
1,92126813,EFA_IDU/T1000/T1000_R1/EFA_IDU_T1000_R1_IMAG00...,T1000,EFA_IDU,2,4e2838a5c2c0294128f8b72b996c2edfcf498b04,B00T11E8080-0000X00S00,B00T11E8080-0000X_0S00,2015-05-22 21:33:46,/data/vision/beery/scratch/antoine/CBM_reid/pr...
2,92126815,EFA_IDU/T1001/T1001_R1/EFA_IDU_T1001_R1_IMAG00...,T1001,EFA_IDU,2,9229c652be571b6970dd2ba7fbe104597dcc851e,B00T11E0000-0000X_0S00,B00T11E____-5000X_0S00,2015-05-22 21:32:50,/data/vision/beery/scratch/antoine/CBM_reid/pr...
3,92126816,EFA_IDU/T1001/T1001_R1/EFA_IDU_T1001_R1_IMAG00...,T1001,EFA_IDU,3,d29ac1e2a0c9da2f40c3f2e1dae60aff100aa042,B00T11E____-0040X_0S00,B00T11E____-5000X_0S00,2015-05-22 21:32:51,/data/vision/beery/scratch/antoine/CBM_reid/pr...
4,92126817,EFA_IDU/T1001/T1001_R1/EFA_IDU_T1001_R1_IMAG00...,T1001,EFA_IDU,4,f73d21ecc60aaf4e940a02283aaeb3ffb4604cee,C00T1_E____-4000X_0S0_,B00T11E____-5000X_0S00,2015-05-22 21:32:52,/data/vision/beery/scratch/antoine/CBM_reid/pr...
